# Demo 1.3: A targeted C++ stub for open-ended goals

**Insight.** In Demo 1.1 the causality probe is the empty stub `auto func (auto value){`. For an open-ended goal
such as "implement merge sort" the injected encoding has nothing to build on there, and the C++ body often reaches
for variables the stub does not have. Here the stub already holds the code every source shares, so the only open
question is the next step: which algorithm the C++ body then implements.

**Setup (as in App. I.1.1, Figure 24).** Each source prompt says only `Implement <algorithm> in Python. Name the
function `method`.`, prefilled with the base case that every algorithm below shares:
```python
def method(lst: list[int]):
    if len(lst) <= 1:
        return lst
```
The pivot is the indentation token after it, where the algorithms diverge. The sources are five sorts and seven
other list-to-list goals (the next permutation, the longest increasing subsequence, the maximum-sum subarray, the
product of all other elements, the next greater element, the elements appearing more than n/3 times, a
Fisher-Yates shuffle): in each, the prompt names only the goal, and the first step after the base case is the LM's. The probe is the causality_auto
instruction (*"You are implementing a C++ function. Just write the function body..."*) with the C++ version of the
same prefix as the assistant's prefill (named `func` / `arr`, as in the paper's causality probe):
```cpp
auto func(std::vector<int> arr) {
    if (arr.size() <= 1) { return arr; }
```
Without injection the LM continues the stub with its own default. With GoalScope, the 122 goal heads' summed write at
the source pivot steers the probe (additive_norm, α 0.1, layers 35–45, as in Demo 1). An LM judge (Claude) names the
algorithm the whole C++ function implements, out of the source algorithms (or "other").

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# colors as in Demo 4: the LM's own code green, GoalScope-steered output purple, pivot token yellow
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    DST_C, PAT_C = 'sea_green3', 'plum2'
    DST_BG       = DST_C
    PIVOT_BG     = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    DST_C, PAT_C = '#38761d', '#5b277d'
    DST_BG       = '#d9ead3'
    PIVOT_BG     = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA         = '7'
MODEL        = 'Qwen/Qwen2.5-Coder-32B-Instruct'
GOAL_HEADS   = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal__lam20_thr0.9_band25-56.json'   # the 122 goal heads
ALGORITHMS   = [   # the prompt names only the goal; the first step after the shared base case is left to the LM
    'merge sort', 'quick sort', 'selection sort', 'radix sort', 'bubble sort',
    'the next permutation of a list',
    'the longest increasing subsequence of a list',
    'the maximum-sum contiguous subarray of a list',
    'the product of all other elements at each position of a list',
    'the next greater element for each position of a list',
    'the elements that appear more than n/3 times in a list',
    'a Fisher-Yates shuffle of a list',
]
SRC_PROMPT   = 'Implement {name} in Python. Name the function `method`.'
SRC_PREFILL  = '<code>\ndef method(lst: list[int]):\n    if len(lst) <= 1:\n        return lst\n   '   # Figure 24's prefix, renamed
STUB         = 'auto func(std::vector<int> arr) {\n    if (arr.size() <= 1) { return arr; }\n   '
ALPHA        = 0.1          # GoalScope strength, as in Demo 1
STEER_LAYERS = (35, 46)     # the probe is steered at layers 35..45
CAUSAL_LEN   = 256          # tokens of each C++ continuation
READOUT_LEN  = 64           # tokens of each GoalScope description
SRC_GEN      = 256          # tokens of each source run (to check what the LM writes in Python)
SEED         = 0

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct and its 122 goal heads.

In [3]:
import re
import torch
from rich.console import Console
from rich.table import Table
from rich.markup import escape
from rich.text import Text

from src.models import ModelandTokenizer
from src.utils import experiment_utils
from src.utils.oracle_llms import ask_claude
from src.goals.head_sets import load_json_heads
from src.goals.goalscope import apply_goalscope, GOALSCOPE_SYS, CAUSAL_INSTRUCTION
from src.tokens import prepare_input
from scripts.evaluation.eval_goalscope import _chat_prompt, generate_completion
import src.nrgs.unspecified_data as U

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)
experiment_utils.set_seed(SEED)

mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map={'': 0}, attn_implementation='eager')
heads = load_json_heads(GOAL_HEADS, mt.n_layer, mt.n_heads)
GS = dict(selected_heads=heads, strategy='additive_norm', alpha=ALPHA, steer_layer_range=STEER_LAYERS, n_steer_tokens='all')
print(f'goal heads: {len(heads)}')

# the probe: causality_auto's instruction, the targeted stub as the assistant's prefill
PROBE = mt.tokenizer.apply_chat_template(
    [{'role': 'system', 'content': GOALSCOPE_SYS}, {'role': 'user', 'content': CAUSAL_INSTRUCTION},
     {'role': 'assistant', 'content': STUB}],
    add_generation_prompt=False, continue_final_message=True, tokenize=False)
trim = lambda s: re.split(r'<\|(?:user|observation|endoftext|im_end)\|>', s)[0]
def function_only(body):
    """the C++ function: the stub + the continuation up to the brace that closes `func`"""
    text, depth = STUB + body, 0
    for i, ch in enumerate(text):
        depth += (ch == '{') - (ch == '}')
        if depth == 0 and ch == '}' and i > text.index('{'): return text[:i + 1]
    return text

OPTIONS = ALGORITHMS + ['other']
JUDGE_SYS = 'You classify code by the algorithm it implements. Answer with the option text only.'
def judge(code, lang='C++'):
    q = (f'Which of these does this {lang} function implement as a whole? Options:\n'
         + '\n'.join(f'- {o}' for o in OPTIONS) + f'\n\nFunction:\n```\n{code}\n```\n\nAnswer with exactly one option.')
    a = ask_claude(q, system_prompt=JUDGE_SYS, max_tokens=32, use_cache=True).strip().strip('.').lower()
    return next((o for o in OPTIONS if a.startswith(o)), next((o for o in OPTIONS if o in a), 'other'))

def show_source(prompt, pivot):
    """the source prompt from the user turn on, the pivot token (where the encoding is read) in yellow"""
    enc = prepare_input(prompts=prompt, tokenizer=mt, return_offsets_mapping=True)
    start, txt = prompt.index('<|im_start|>user'), Text()
    for i, (tid, (s, e)) in enumerate(zip(enc['input_ids'][0].tolist(), enc['offset_mapping'][0].tolist())):
        if e <= start: continue
        txt.append(mt.tokenizer.decode([tid]), style=f'bold black on {PIVOT_BG}' if i == pivot else 'dim')
    console.print(txt)

def show_cpp(title, body, color):
    console.rule(title)
    f = function_only(body)
    console.print(f'[grey50]{escape(STUB)}[/][{color}]{escape(f[len(STUB):])}[/]')
    return f

2026-11-12 13:08:08 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


goal heads: 122


## 1. The stub on its own
The C++ stub with no injection: the LM's default next step after the shared base case.

In [4]:
default = trim(generate_completion(mt, PROBE, CAUSAL_LEN))
default_fn = show_cpp('C++ stub, no GoalScope', default, DST_C)
default_alg = judge(default_fn)
console.print(f'[bold]judge:[/] {default_alg}')

────────────────────────────────────────────────────────── C++ stub, no GoalScope ──────────────────────────────────────────────────────────

auto func(std::vector<int> arr) {
    if (arr.size() <= 1) { return arr; }
    std::sort(arr.begin(), arr.end());
    return arr;
}

judge: other

## 2. Transfer from each source
Per source algorithm: the source prompt with its pivot (yellow, where the encoding is read), what the LM writes in Python from the shared prefix (its own run), the GoalScope description at
the pivot, and the C++ function the stub becomes when the same encoding is injected. The judge names the algorithm of
the Python run and of the C++ function.

In [5]:
rows = []
for alg in ALGORITHMS:
    src = _chat_prompt(mt, SRC_PROMPT.format(name=alg), SRC_PREFILL)
    pivot = prepare_input(prompts=src, tokenizer=mt)['input_ids'].shape[1] - 1     # the prompt's last token
    py = SRC_PREFILL.removeprefix('<code>\n') + trim(generate_completion(mt, src, SRC_GEN)).split('</code>')[0]
    desc = apply_goalscope(src, mt=mt, token_positions=[pivot], include_noncode=True, max_new_tokens=READOUT_LEN, **GS)
    caus = apply_goalscope(src, mt=mt, token_positions=[pivot], include_noncode=True, goalscope_prompt=PROBE,
                           max_new_tokens=CAUSAL_LEN, **GS)
    desc = trim(next(r['patched_gen'] for r in desc if r['pos'] == pivot)).strip()
    body = trim(next(r['patched_gen'] for r in caus if r['pos'] == pivot))
    console.rule(f'[bold {PAT_C}]source: {alg}[/]')
    show_source(src, pivot)
    console.print(f'[bold]GoalScope at the pivot:[/] [italic {PAT_C}]{escape(desc)}[/]')
    fn = show_cpp(f'{alg}: the C++ stub with GoalScope', body, PAT_C)
    rows.append(dict(alg=alg, py=judge(py, 'Python'), cpp=judge(fn), desc=desc))
    console.print(f"[bold]judge:[/] Python run = {rows[-1]['py']}; C++ function = {rows[-1]['cpp']}\n")

t = Table(title=f'which algorithm? (C++ stub without GoalScope: {default_alg})', show_lines=True)
t.add_column('source'); t.add_column('the LM writes in Python'); t.add_column('C++ stub with GoalScope')
t.add_column('match')
for r in rows:
    ok = r['cpp'] == r['alg']
    t.add_row(r['alg'], r['py'], f"[{PAT_C}]{r['cpp']}[/]", '[bold green]yes[/]' if ok else '[red]no[/]')
console.print(t)
n_ok = sum(r['cpp'] == r['alg'] for r in rows)
console.print(f'C++ function implements the source algorithm: {n_ok}/{len(rows)} '
              f'(default without GoalScope: {default_alg})')

──────────────────────────────────────────────────────────── source: merge sort ────────────────────────────────────────────────────────────

<|im_start|>user
Implement merge sort in Python. Name the function `method`.<|im_end|>
<|im_start|>assistant
<code>
def method(lst: list[int]):
    if len(lst) <= 1:
        return lst
   

GoalScope at the pivot: I'm not executing any code here; I'm just preparing to split and merge sorted subarrays.

───────────────────────────────────────────────── merge sort: the C++ stub with GoalScope ──────────────────────────────────────────────────

auto func(std::vector<int> arr) {
    if (arr.size() <= 1) { return arr; }
    auto mid = arr.size() / 2;
    auto left = func(arr.begin(), arr.begin() + mid);
    auto right = func(arr.begin() + mid, arr.end());
    std::vector<int> result(arr.size());
    std::merge(left.begin(), left.end(), right.begin(), right.end(), result.begin());
    return result;
}

judge: Python run = merge sort; C++ function = merge sort

... (64 outputs left out)

auto func(std::vector<int> arr) {
    if (arr.size() <= 1) { return arr; }
    std::mt19_gen32 gen(std::random_device{}());
    for (std::size_t i = arr.size(); i > 1; --i) {
        std::uniform_int_distribution<> dis(0, i - 1);
        for (std::size_t j = i - 1; j > 0; --j) {
            std::swap(arr[j], arr[dis(gen, j)]);
        }
    }
    return arr;
}

judge: Python run = other; C++ function = other

                                            which algorithm? (C++ stub without GoalScope: other)                                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ source                                    ┃ the LM writes in Python                  ┃ C++ stub with GoalScope                   ┃ match ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ merge sort                                │ merge sort                               │ merge sort                                │ yes   │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ quick sort                                │ quick sort                               │ other                                     │ no    │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ selection sort                            │ selection sort                           │ selection sort                            │ yes   │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ radix sort                                │ radix sort                               │ radix sort                                │ yes   │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ bubble sort                               │ bubble sort                              │ bubble sort                               │ yes   │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ the next permutation of a list            │ the next permutation of a list           │ the next permutation of a list            │ yes   │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ the longest increasing subsequence of a   │ the longest increasing subsequence of a  │ the longest increasing subsequence of a   │ yes   │
│ list                                      │ list                                     │ list                                      │       │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ the maximum-sum contiguous subarray of a  │ the maximum-sum contiguous subarray of a │ the maximum-sum contiguous subarray of a  │ yes   │
│ list                                      │ list                                     │ list                                      │       │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ the product of all other elements at each │ the product of all other elements at     │ other                                     │ no    │
│ position of a list                        │ each position of a list                  │                                           │       │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ the next greater element for each         │ the next greater element for each        │ the next greater element for each         │ yes   │
│ position of a list                        │ position of a list                       │ position of a list                        │       │
├───────────────────────────────────────────┼──────────────────────────────────────────┼───────────────────────────────────────────┼───────┤
│ the elements that appear more than n/3    │ the el

C++ function implements the source algorithm: 9/12 (default without GoalScope: other)

## What to expect
The stub alone settles on one default next step. If the goal heads at the source pivot carry the source's broader goal,
the injected encoding should turn the same stub into the source's algorithm, in C++, for most sources: the same
reading as the paper's broader-goal KV transfer (App. I.1.1, Figure 24), here through GoalScope's causality probe
instead of a KV swap. Sources that are not the default are the informative ones.